# NumFast — тест-драйв новой архитектуры

Контейнерный движок (Row-Struct), ранняя/поздняя компрессия, JIT WGSL-шейдеры.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

from _core.backend import get_xp, get_active_name, set_active
backend = get_active_name()
xp = get_xp()
print(f"Бэкенд вычислений: {backend}")
print(f"Библиотека массивов: {xp.__name__} {getattr(xp, '__version__', '')}")
if backend == 'wgpu':
    print("  → GPU-шейдеры (WGSL), хост-массивы на numpy")
elif backend == 'cupy':
    print(f"  → GPU: {get_device()}")
else:
    print("  → CPU-режим (numpy)")

Бэкенд вычислений: wgpu
Библиотека массивов: numpy 2.4.6
  → GPU-шейдеры (WGSL), хост-массивы на numpy


## 1. Контекст и ядро

In [2]:
from _core.context import create_context
from _core import kernel

kernel.clear_all()
ctx = create_context("demo")
print(f"Context ID: {ctx['_id']}")

Context ID: e0dd58a591d7426bad01849a4c59d6ef


## 2. Stats — единый диспетчер моментов

Работает на сырых списках, Series, ColumnProxy через JIT WGSL:

In [3]:
from Stats.Stats import total, minimum, maximum, mean, var, std, all_stats

data = [10.0, 20.0, 30.0, 40.0, 50.0]
print(f"total  = {total(data)}")
print(f"mean   = {mean(data)}")
print(f"var    = {var(data, ddof=1)}")
print(f"std    = {std(data, ddof=0)}")
print(f"min    = {minimum(data)}")
print(f"max    = {maximum(data)}")

total  = 150.0
mean   = 30.0
var    = 250.0
std    = 14.142135623730951
min    = 10.0
max    = 50.0


## 3. Series — скрытая контейнерная таблица

make_series создаёт одно-колоночную Row-Struct таблицу и возвращает ColumnProxy:

In [4]:
from _core.series import make_series

s = make_series([1.0, 2.0, 3.0, 4.0, 5.0], ctx)
print(f"Series ID:  {s['_series_id']}")
print(f"Table ID:   {s['_table_id']}")
print(f"Col name:   {s['_col_name']}")
print(f"Length:     {s['_length']}")
print(f"Hidden:     {s.get('_is_hidden_series')}")
print(f"Stats via proxy: total = {total(s)}")

Series ID:  tbl_1
Table ID:   tbl_1
Col name:   _value
Length:     5
Hidden:     True
Stats via proxy: total = 15.0


In [5]:
print()
s.info()


#,Column,Dtype,Bits,Packing,Compression,Memory
0,_value,float32,32,part0[0:32),none,20 B


In [6]:
from _core.series import series_add, series_sub

a = make_series([10, 20, 30], ctx)
b = make_series([1, 2, 3], ctx)
c = series_add(a, b)
print(f"add: total(c) = {total(c)}, mean(c) = {mean(c)}")
d = series_sub(a, b)
print(f"sub: total(d) = {total(d)}, mean(d) = {mean(d)}")

add: total(c) = 66.0, mean(c) = 22.0
sub: total(d) = 54.0, mean(d) = 18.0


## 4. Row-Struct Container — упаковка в битовые строки

Колоночные данные пакуются в строки фиксированной битовой ширины без промежутков:

In [7]:
from _core.container import compute_layout, pack_rows, extract_column

schema = [
    {"name": "price", "dtype": "float32"},
    {"name": "rsi",   "dtype": "scaled", "bit_width": 16,
     "scale": 0.01, "offset": 0.0, "min_int": -32768.0},
]
data = {
    "price": [100.0, 101.5, 99.0, 102.0, 98.5],
    "rsi":   [70.0, 55.0, 30.0, 65.0, 40.0],
}

packed = pack_rows(schema, data)
print(f"Строк: {packed['num_rows']}, частей: {packed['num_parts']}")
print(f"Сырые строки: {packed['rows'][:2]}...")  # первые 2 строки

extracted = extract_column(packed['rows'], 'price', packed['layout'])
print(f"price (raw f32): {extracted}")
extracted_rsi = extract_column(packed['rows'], 'rsi', packed['layout'])
print(f"rsi (scaled):    {extracted_rsi}")

Строк: 5, частей: 2
Сырые строки: [[1120403456, 39768], [1120600064, 38268]]...
price (raw f32): [100.0, 101.5, 99.0, 102.0, 98.5]
rsi (scaled):    [70.0, 55.0, 30.0, 65.0, 40.0]


## 5. Многоколоночные таблицы + ColumnProxy + Stats

Через Tables._lib создаём контейнерную таблицу и получаем column proxy:

In [8]:
from Tables._lib.tables_lib import _container_table_create, _get_column

kernel.clear_all()
schema = [
    {"name": "open",  "dtype": "float32"},
    {"name": "high",  "dtype": "float32"},
    {"name": "low",   "dtype": "float32"},
    {"name": "close", "dtype": "float32"},
]
data = {
    "open":  [100.0, 101.0, 99.5],
    "high":  [102.0, 103.0, 100.5],
    "low":   [99.0, 100.0, 98.0],
    "close": [101.0, 102.5, 99.0],
}
tbl = _container_table_create(schema, data)

for name in ["open", "high", "low", "close"]:
    proxy = _get_column(tbl, name)
    m = all_stats(proxy)
    print(f"{name:6s}: sum={m['sum']:.1f}, mean={m['mean']:.2f}, min={m['min']:.1f}, max={m['max']:.1f}")

open  : sum=300.5, mean=100.17, min=99.5, max=101.0
high  : sum=305.5, mean=101.83, min=100.5, max=103.0
low   : sum=297.0, mean=99.00, min=98.0, max=100.0
close : sum=302.5, mean=100.83, min=99.0, max=102.5


## 6. Ранняя компрессия float32 → N-бит scaled

В схеме можно указать `compression: {scaled: True, bits, scale, offset}` — pack_rows применит сжатие:

In [9]:
kernel.clear_all()
schema = [
    {"name": "price", "dtype": "float32",
     "compression": {"scaled": True, "bits": 12, "scale": 0.1, "offset": 100.0}},
    {"name": "volume", "dtype": "float32"},
]
data = {
    "price": [150.0, 200.0, 175.0],
    "volume": [1000, 2000, 1500],
}

from _core.container import compute_layout
layout = compute_layout(schema)
c = layout['columns']['price']
print(f"price: size={c['size']} bit, parts={c['parts']}")

tbl = _container_table_create(schema, data)
price_proxy = _get_column(tbl, 'price')
vol_proxy = _get_column(tbl, 'volume')

print(f"price total  = {total(price_proxy):.2f}  (ожидается ~525.0)")
print(f"volume total = {total(vol_proxy):.0f}  (ожидается 4500)")

price: size=12 bit, parts=[(0, 0, 12)]
price total  = 525.00  (ожидается ~525.0)
volume total = 4500  (ожидается 4500)


In [10]:
print()
tbl.info()


#,Column,Dtype,Bits,Packing,Compression,Memory
0,price,float32,12,part0[0:12),"compressed, scale=0.1, offset=100.0",6 B
1,volume ↗,float32,32,part0[12:32) + part1[0:12),none,12 B


## 7. Динамическая (поздняя) компрессия

Сжимаем существующую float32 колонку в таблице на CPU или через JIT WGSL:

In [11]:
kernel.clear_all()
schema = [{"name": "val", "dtype": "float32"}]
raw_data = {"val": [50.0, 51.0, 52.0, 53.0, 54.0]}
tbl = _container_table_create(schema, raw_data)

from Tables._lib.tables_lib import _compress_column
# Сжимаем 32-bit → 12-bit scaled: scale=0.1, offset=50.0
compressed = _compress_column(tbl, "val", target_bits=12, scale=0.1, offset=50.0)

proxy = _get_column(compressed, "val")
print(f"После компрессии: total = {total(proxy):.2f}")
print(f"  (ожидается ~260.0)")
print()
print(f"Было частей: {tbl['_num_parts']}, стало: {compressed['_num_parts']}")

После компрессии: total = 260.00
  (ожидается ~260.0)

Было частей: 1, стало: 1


## 8. Scaled Series через контейнер

Старая `make_series(compression=...)` теперь создаёт скрытую таблицу со scaled-колонкой:

In [12]:
kernel.clear_all()

data = [10.0, 20.0, 30.0, 40.0, 50.0]
comp = {"bits": 16, "scale": 40.0/65535.0, "offset": 10.0}
s = make_series(data, ctx, compression=comp)

print(f"total(s) = {total(s):.2f}  (ожидается 150.0)")
print(f"mean(s)  = {mean(s):.2f}  (ожидается 30.0)")
print()
s.info()


total(s) = 150.00  (ожидается 150.0)
mean(s)  = 30.00  (ожидается 30.0)



#,Column,Dtype,Bits,Packing,Compression,Memory
0,_value,float32,16,part0[0:16),"compressed, scale=0.0006103608758678569, offset=10.0",10 B


## 9. Компрессия напрямую — pack / unpack

Модуль `_core.compression` для ручного управления:

In [13]:
from _core.compression import pack_scaled, unpack_scaled, pack_enum, unpack_enum

scaled_packed, meta = pack_scaled([10.0, 20.0, 30.0], "int8")
print(f"pack_scaled: {scaled_packed}, meta={meta}")
restored = unpack_scaled(scaled_packed, meta)
print(f"unpack: {restored}")

enum_packed, emeta = pack_enum(["AAPL", "MSFT", None, "TSLA"])
print(f"pack_enum:  {enum_packed}, vocab={emeta['vocab']}")
erestored = unpack_enum(enum_packed, emeta)
print(f"unpack: {erestored}")

pack_scaled: [-128, 0, 127], meta={'type': 'scaled', 'scale': 0.0784313725490196, 'offset': 10.0, 'min_int': -128.0, 'dtype': 'int8', 'min': 10.0, 'max': 30.0}
unpack: [10.0, 20.03921568627451, 30.0]
pack_enum:  [0, 1, 2, 3], vocab={0: 'AAPL', 1: 'MSFT', 3: 'TSLA'}
unpack: ['AAPL', 'MSFT', None, 'TSLA']


## 10. Регистрация таблицы в Kernel

Прямая регистрация и извлечение:

In [14]:
kernel.clear_all()
schema = [{"name": "x", "dtype": "float32"}]
packed = pack_rows(schema, {"x": [1.0, 2.0, 3.0]})
tid = kernel._register_table(packed["rows"], packed["num_parts"],
                           packed["schema"], packed["layout"])
entry = kernel._get_table_entry(tid)
print(f"Table ID: {tid}")
print(f"Rows: {entry['rows']}")
print(f"Num rows: {entry['num_rows']}")

extracted = extract_column(entry["rows"], "x", entry["layout"])
print(f"Extracted: {extracted}")

Table ID: tbl_1
Rows: [[1065353216], [1073741824], [1077936128]]
Num rows: 3
Extracted: [1.0, 2.0, 3.0]


## 11. JIT WGSL — прямо на GPU

Если бэкенд — wgpu, stats на ColumnProxy исполняются через сгенерированный шейдер:

In [15]:
print(f"Активный бэкенд: {get_active_name()}")

if get_active_name() == 'wgpu':
    from _core.backend._wgpu import generate_column_moments_shader
    schema = [
        {"name": "a", "dtype": "float32"},
        {"name": "b", "dtype": "float32"},
    ]
    layout = compute_layout(schema)
    shader = generate_column_moments_shader(layout, "b")
    print("Сгенерированный WGSL шейдер для колонки 'b':")
    print("---")
    print(shader[:400] + "...")
else:
    print("WGSL недоступен — Stats работают через CPU (extract_column + numpy)")

Активный бэкенд: wgpu
Сгенерированный WGSL шейдер для колонки 'b':
---

struct Row {
    part0: u32,
    part1: u32,
};

struct Params {
    valid_elements: u32,
    _pad0: u32,
    _pad1: u32,
    _pad2: u32,
    _pad3: u32,
    _pad4: u32,
    _pad5: u32,
    _pad6: u32,
};

@group(0) @binding(0) var<storage, read> input: array<Row>;
@group(0) @binding(1) var<storage, read_write> output: array<f32>;
@group(0) @binding(2) var<storage, read> params: Params;

fn load_...


## 12. Полный цикл: сжатие → контейнер → Stats → WGSL

In [16]:
kernel.clear_all()
import numpy as np

# Original
np.random.seed(42)
n = 1000
prices = list(np.random.uniform(50, 150, n).astype(float))
volumes = list(np.random.uniform(1000, 10000, n).astype(float))

schema = [
    {"name": "price", "dtype": "float32",
     "compression": {"scaled": True, "bits": 12, "scale": 0.1, "offset": 50.0}},
    {"name": "volume", "dtype": "float32"},
]
tbl = _container_table_create(schema, {"price": prices, "volume": volumes})

price_proxy = _get_column(tbl, "price")
vol_proxy = _get_column(tbl, "volume")

p_stats = all_stats(price_proxy)
v_stats = all_stats(vol_proxy)
print(f"price: sum={p_stats['sum']:.1f}, mean={p_stats['mean']:.2f}, "
      f"min={p_stats['min']:.1f}, max={p_stats['max']:.1f}")
print(f"volume: sum={v_stats['sum']:.1f}, mean={v_stats['mean']:.2f}, "
      f"min={v_stats['min']:.1f}, max={v_stats['max']:.1f}")

print()
tbl.info()

if get_active_name() == "wgpu":
    print("\n✅ JIT WGSL shader worked — data never left GPU")
else:
    print("\n⚠️  CPU path — extract_column + numpy")


price: sum=99024.7, mean=99.02, min=50.5, max=150.0
volume: sum=5563156.0, mean=5563.16, min=1029.0, max=9994.7



#,Column,Dtype,Bits,Packing,Compression,Memory
0,price,float32,12,part0[0:12),"compressed, scale=0.1, offset=50.0",2.0 KB
1,volume ↗,float32,32,part0[12:32) + part1[0:12),none,3.9 KB



✅ JIT WGSL shader worked — data never left GPU
